## Comments
This uses the output of the "MACAW_tests" script and assumes the filenames created by these tests.  
Change the naming convention in the get_all_duplicate_pairs to fit yours.  
Additionally you may need to change the use of the function in Main if you didn't included the model id in the file names

## Imports

In [1]:
import cobra
from cobra.io import read_sbml_model, write_sbml_model
import os
import pandas as pd

## Paths

In [2]:
# Path to your models
model_dir = '/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr/'
model_dir_save = '/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp/'
# Path to the .csv files created by the "MACAW_Phase_1_tests" script
test_results_dir = '/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/MACAW_results/'

## Functions

In [3]:
# Summarizes the test results for the diphosphate test throughout all files in the test_results_dir
def create_diphsophate_overview(test_results_dir):

    # Create lists to store the reactions that should be irreversible and those that should be flipped and made irreversible
    irreversible = list()
    flipped_and_irr = list()

    # Iterate through the test results directory and read each CSV file
    for file in os.listdir(test_results_dir):
        if file.endswith(".csv"):
            df = pd.read_csv(test_results_dir+file)

            # Create smaller df that only contain the rows with the columns of interest
            # and put all ids in the corresponding list
            irreversible_df = df[df.iloc[:, 6] == 'should be irreversible'] 
            irreversible.extend(irreversible_df.iloc[:, 0].tolist())
            flipped_and_irr_df = df[df.iloc[:, 6] == 'should be flipped and made irreversible']
            flipped_and_irr.extend(flipped_and_irr_df.iloc[:, 0].tolist())

    # Remove duplicates from the lists
    irreversible = list(set(irreversible))
    flipped_and_irr = list(set(flipped_and_irr))

    # Print the results
    print(f"{len(irreversible)} reactions should be Irreversible reactions:")
    for i in irreversible:
        print(i)
    print("-------------------------------------------------------------------------\n")
    print(f"{len(flipped_and_irr)} reactions should be Flipped and irreversible reactions:")
    for i in flipped_and_irr:
        print(i)

    return irreversible, flipped_and_irr

In [4]:
# MACAW flags all diphosphate containing reactions that are reversible, since these are typically irreversible.
# After assessing the flagged reactions provided by the create_diphsophate_overview() function (cell above),
# this function sets new bounds for the flagged reactions based on their assessment
def fix_diphophate_reactions():

    # Contrain all reactions that should be irreversible
    for reaction in make_irreversible_ppi:
        if reaction in model.reactions:
            r_obj = model.reactions.get_by_id(reaction)
            try:
                if check_influence_reversibility(model, r_obj):
                                r_obj.bounds = (-1000.0, 1000.0)
                                print(f"Kept {reaction} reversible in {model.id} due to objective value impact.")
                else:
                    r_obj.bounds = (0.0, 1000.0)
                    print(f"Fixed {reaction} to forward-irreversible (0.0, 1000.0) in {model.id}")
            except KeyError:
                continue
    
    # Constrain all reactions that should be irreversible with the diphosphat being a reactant (on the left side in the reaction string)
    for reaction in flip_and_make_irreversible_ppi:
        try:
            model.reactions.get_by_id(reaction).bounds = (-1000.0, 0.0)
            print(f'Fixed {reaction} in {model}')
        except KeyError:
            continue

In [5]:
# Checks if altering a reaction's bounds significantly impacts the objective value.
# Returns True if the reaction should REMAIN REVERSIBLE, False otherwise.

def check_influence_reversibility(model, rxn_id, threshold_pct=0.1):
    if rxn_id not in model.reactions:
        return False
    with model:
        #check reversibility of reactions by comparing objective value pre and post reversibility 
        sol1 = model.optimize()
        sol1_val = sol1.objective_value if sol1.status == 'optimal' else 0.0
        rxn = model.reactions.get_by_id(rxn_id)
        rxn.bounds = (-1000.0, 1000.0)
        sol2 = model.optimize()
        sol2_val = sol2.objective_value if sol2.status == 'optimal' else 0.0

        if sol1_val == 0.0:
            print(f"ATTENTION {model.id}: Restricting {rxn_id} results in ZERO growth.")
            return True 

        diff_objv = abs(sol2_val - sol1_val)
        threshold = threshold_pct * sol1_val
        
        if diff_objv > threshold:
            print(f"{model.id}: Changing {rxn_id} significantly alters objective value by: {diff_objv:.4f}")
            return True  

    return False 

In [6]:
# Use the .csv files from the phase_1 MACAW tests to create a list of all 'duplicate pairs' for a model
# This will be four lists of tuples that represent all reactions in the model that where flagged by 
# MACAW as an exact-, directional-, coefficient- or redox-duplicate

# This takes the model_id as input (without any fomrat suffix)
def get_all_duplicate_pairs(model_id):

    # The name of the test_result file corresponding to this model is assumed and the csv is read
    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!CHANGE THIS DEPENDING ON YOUR NAMING CONVENTION!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    test_results = pd.read_csv(test_results_dir+f"{model_id}_macaw.csv")

    # Filter reactions with entries not equal to "ok" in the specified columns
    duplicates_df = test_results.loc[
        (test_results['duplicate_test_exact'] != 'ok') |
        (test_results['duplicate_test_directions'] != 'ok') |
        (test_results['duplicate_test_coefficients'] != 'ok') |
        (test_results['duplicate_test_redox'] != 'ok'),
        ['reaction_id', 'duplicate_test_exact', 'duplicate_test_directions', 'duplicate_test_coefficients', 'duplicate_test_redox']
    ]

    # Create a dict containing the model reactions as keys and the test results for each test as value (dict)
    # This will contain all reactions that were flagged by at least one duplicate test
    model_duplicates_dict = {
        row['reaction_id']: {
            "Exact": row['duplicate_test_exact'],
            "Directions": row['duplicate_test_directions'],
            "Coefficient": row['duplicate_test_coefficients'],
            "Redox": row['duplicate_test_redox']
        }
        for _, row in duplicates_df.iterrows()
    }

    # Create lists for all types of duplicates
    exact_duplicates = list()
    directional_duplicates = list()
    coefficient_duplicates = list()
    redox_duplicates = list()

    # Iterate through all reactions and corresponding duplicate test results
    for reaction_id, tests in model_duplicates_dict.items():

        # Append all reactions flagged by MACAW and the corresponding duplicate reactions
        if tests['Exact'] != 'ok':
            exact_duplicates.append((reaction_id, tests['Exact']))
        if tests['Directions'] != 'ok':
            directional_duplicates.append((reaction_id, tests['Directions']))
        if tests['Coefficient'] != 'ok':
            coefficient_duplicates.append((reaction_id, tests['Coefficient']))
        if tests['Redox'] != 'ok':
            redox_duplicates.append((reaction_id, tests['Redox']))

    
    # Apply the format_duplicate_pairs function to split reactions with multiple duplicates for the same type
    # into singular pairs (see function below)
    exact_duplicates = formate_duplicate_pairs(exact_duplicates)
    directional_duplicates = formate_duplicate_pairs(directional_duplicates)
    coefficient_duplicates = formate_duplicate_pairs(coefficient_duplicates)
    redox_duplicates = formate_duplicate_pairs(redox_duplicates)

    # Returns all 4 lists 
    return exact_duplicates, directional_duplicates, coefficient_duplicates, redox_duplicates

In [7]:
# A function to create singular reaction pairs in all duplicate lists
# This is necessary since some reactions have several duplicate reactions

# The function takes a list of tuples as input
def formate_duplicate_pairs(duplicate_list):

    # Create a new duplicate list for the formated results at the end = output list
    formated_duplicate_list = list()

    # Iterate through all pairs in the input list and check if there are multiple duplicates for the same reaction
    # This is indicated by the presence of a semicolon
    for r1, r2 in duplicate_list:

        # If a semicolon is found new pairs will be created and added to the output list
        if ";" in r2:

            # Split the string by semicolon and create new pairs
            r2_list = r2.split(";")

            # Create a list for new pairs
            new_pairs = list()
            
            # Create new pairs from all duplicates and add them to the new_pairs list
            # The new pairs will be sorted to later remove doubles
            for r in r2_list:
                new_pairs.append(tuple(sorted((r1, r))))

            # Extend the output list with new pairs
            formated_duplicate_list.extend(new_pairs)

        # When no semicolon is found the pair is sorted and just added to the output list
        else:
            formated_duplicate_list.append(tuple(sorted((r1, r2))))

    # Remove all duplucate pairs (so all pairs of duplicates that are in the list multiple times)
    formated_duplicate_list = list(set(formated_duplicate_list))

    # Return the formated list of duplicates 
    # This is also a list of tuples and most likely longer than the input
    return formated_duplicate_list

In [8]:
# Creates a dictionary containing all reactions in the model as keys
# The value is always a list of the sorted genes corresponding to this reaction 
# OR ["None"] if no GPR could be found for this reaction in this model

# Input is a cobra model object
def create_gpr_list_dict(model):

    # Create the output dict
    gpr_list_dict = {}

    # Iterate through all reactions in the model
    for reaction in model.reactions:

        # Get the GPR as string
        gpr_string = reaction.gene_reaction_rule

        # If no GPR is given set ["None"] as the value in the output dict
        if gpr_string == '':
            gpr_list_dict[reaction.id] = ["None"]

        # If GPR are found formate them into a list by first splitting them by the 'or' relation
        # Genes with an 'and' relation will be split further and added to the GPR list as an list themself
        else:
            gpr_list_or_split = list(gpr_string.split(' or '))

            # Create a list where both 'or' and 'and' relations are kept = Value list
            gpr_list_or_and_split = []

            # Check if an entry in the list contains an 'and'
            for entry in gpr_list_or_split:

                # If yes the split this entry into a new list, clean it and add the list to the value list
                if 'and' in entry:
                    new_entry = list(entry.split(' and '))
                    new_entry_cleaned = [s.replace("(", "").replace(")", "") for s in new_entry]

                    # Sort this sublist alphabetically
                    gpr_list_or_and_split.append(sorted(new_entry_cleaned))

                # If no then just append the entry into the output list
                else:
                    gpr_list_or_and_split.append(entry)

            # Sort the output list alphabetically by treating each entry as a string 
            # Add the sorted value list to the output dict
            gpr_list_dict[reaction.id] = sorted(gpr_list_or_and_split, key=lambda x: ' '.join(x) if isinstance(x, list) else x)

    # Output is a dict with an entry for each reaction in the input model
    return gpr_list_dict

In [9]:
# A function to compare GPR lists created by the create_gpr_list_dir function
# The output is highly variable and specific
# Only strings describing the relation of the GPR for both input reactions are returned

# Input are two reaction_ids
def compare_gpr_lists(r1, r2):
    if (r1 not in gpr_list_dict) or (r2 not in gpr_list_dict):
        return 'One or both reactions missing from GPR dict'
    # GPR lists for both input reactions are fetched from the dir
    r1_gpr = gpr_list_dict[r1]
    r2_gpr = gpr_list_dict[r2]

    # If either of the reactions has no GPR it is checked if they are both boundary reactions or not
    if (r1_gpr[0] == "None") or (r2_gpr[0] == "None"):
        if (model.reactions.get_by_id(r1) in model.boundary) and (model.reactions.get_by_id(r2) in model.boundary):
            return 'No GPR and both boundary reactions'
        else:
            return 'No GPR but one not a boundary reaction'
    
    # Do they have the same GPR list?
    elif r1_gpr == r2_gpr:
        return 'Same GPR list'
    
    # Is the list the same lenght and shares genes or not?
    elif len(r1_gpr) == len(r2_gpr):
        if any(item in r1_gpr for item in r2_gpr):
            return 'Same length of GPR list and share some genes'
        else:
            return 'Same length of GPR list but no common genes'
    
    # Are the lists of different lenght and:
    # Is one list a sublist of the other?
    # Share the lists genes or not?
    else:
        if (len(r1_gpr) > len(r2_gpr)) and all(item in r1_gpr for item in r2_gpr):
            return 'r2 in r1'
        elif (len(r2_gpr) > len(r1_gpr)) and all(item in r2_gpr for item in r1_gpr):
            return 'r1 in r2'
        elif any(item in r1_gpr for item in r2_gpr):
            return 'Different lenght of GPR list and share some genes'
        else:
            return 'Different lenght of GPR list but no common genes'
    

In [10]:
# Lists containing both lists and strings cannot be made unique with set()  
# Therefore here a function doing that
# Requires that lists tat are items in this list are sorted
def remove_duplicates_in_mixed_list(list):

    # To keep track of checked list items
    seen = set()

    # Output list with duplicates filtered out
    unique_items = []

    # Check each item and treat it depending on its type
    for item in list:
        # Try to convert to tuple (for lists), otherwise leave as-is (for strings)
        try:
            key = tuple(item)
        except TypeError:
            key = item

        # Check if item was alredy checked and append it if not
        if key not in seen:
            seen.add(key)
            unique_items.append(item)

    return unique_items

In [11]:
# Exact duplicates will be removed or combined based on their GPR 

def fix_exact_duplicates():

    # Compare the GPR lists of both reactions and apply fixes depending on the returned status
    for r1, r2 in exact_duplicates:
        if (r1, r2) in rxns2keep or (r2, r1) in rxns2keep:
            print(r1,r2)
            return
        status = compare_gpr_lists(r1, r2)
        if status == 'One or both reactions missing from GPR dict':
            return

        # Delete one reaction if they have the same GPR
        if status == "Same GPR list":
            if r2 in model.reactions:
                model.remove_reactions([model.reactions.get_by_id(r2)])
            else:
                print(f"Notice: Reaction '{r2}' was flagged for deletion but is already removed.")
        
        # If the GPR list of r2 is a substring of the r1 GPR list, delete r2
        elif status == "r2 in r1":
            if r2 in model.reactions:
                model.remove_reactions([model.reactions.get_by_id(r2)])
            else:
                print(f"Notice: Reaction '{r2}' was flagged for deletion but is already removed.")


        # If the GPR list of r1 is a substring of the r2 GPR list, delete r1
        elif status == "r1 in r2":
            if r1 in model.reactions:
                model.remove_reactions([model.reactions.get_by_id(r1)])
            else:
                print(f"Notice: Reaction '{r1}' was flagged for deletion but is already removed.")

        # If both reactions share genes in their GPR list then combine the the reactions by
        # adding all GPR of r2 to r1 and set the new list as the gpr string for r1
        # Then delete r2
        elif (status == 'Same length of GPR list and share some genes') or (status == 'Different length of GPR list and share some genes'):
            new_gpr_list = remove_duplicates_in_mixed_list(gpr_list_dict[r1] + (gpr_list_dict[r2]))
            new_gpr_string = create_gpr_strings(new_gpr_list)
            model.reactions.get_by_id(r1).gene_reaction_rule = new_gpr_string
            model.remove_reactions([model.reactions.get_by_id(r2)])
    
        # For all status where no fix was found yet
        else:
            print(f'No fix possible for {r1} and {r2} with status: {status}')
            return

        print(f"Fixed reactions {r1} and {r2} with status {status}")


In [12]:
# A function to transform a GPR list back into a GPR string 

def create_gpr_strings(gpr_list):
    result = []
    for item in gpr_list:
        if isinstance(item, list):
            # Join sublist with ' and ' and wrap in parentheses
            sub_str = ' and '.join(item)
            result.append(f'({sub_str})')
        else:
            result.append(item)
    return ' or '.join(result)

In [13]:
# Directional duplicates will be removed or kept based on their GPR

def fix_directional_duplicates():

    # Compare the GPR lists of both reactions and apply fixes depending on the returned status
    for r1, r2 in directional_duplicates:
        status = compare_gpr_lists(r1, r2)

        # Delete one reaction based on the directionality of both if the GPR list is the same
        # Always keep the reversible reaction or make r1 reversible if both are irreversible and delete r2
        if status == "Same GPR list":
            if model.reactions.get_by_id(r1).reversibility:
                model.remove_reactions([model.reactions.get_by_id(r2)])
            elif model.reactions.get_by_id(r2).reversibility:
                model.remove_reactions([model.reactions.get_by_id(r1)])
            else:
                model.reactions.get_by_id(r1).bounds = (1000.0, 1000.0)
                model.remove_reactions([model.reactions.get_by_id(r2)])

        # If the GPR list of r2 is a substring of the r1 GPR list and r1 is reversible, delete r2
        elif status == "r2 in r1":
            if model.reactions.get_by_id(r1).reversibility:
                model.remove_reactions([model.reactions.get_by_id(r2)])

        # If the GPR list of r1 is a substring of the r2 GPR list and r2 is reversible, delete r1
        elif status == "r1 in r2":
            if model.reactions.get_by_id(r2).reversibility:
                model.remove_reactions([model.reactions.get_by_id(r1)])

        # For all status where no fix was found yet
        else:
            print(f'No fix possible for {r1} and {r2} with status: {status}')
            return

        print(f"Fixed reactions {r1} and {r2} with status {status}")

In [14]:
def manual_delete_rxns(model, list_of_rxns):
    for rxn_id in list_of_rxns:
        if rxn_id in model.reactions:
            sol1 = model.slim_optimize()
            
            with model:
                model.remove_reactions([model.reactions.get_by_id(rxn_id)])
                sol2 = model.slim_optimize()
                print(f"{rxn_id} Test - Before: {sol1:.4f}, After: {sol2:.4f}")
            
            # If sol2 is less than 10% of sol1, it means rxn is CRITICAL for growth -> Keep it.
            if sol2 < (sol1 * 0.1):
                print(f"{rxn_id} kept: It is required for biomass growth.")
            else:
                # Otherwise, growth is fine without it -> Delete it permanently.
                model.remove_reactions([model.reactions.get_by_id(rxn_id)])
                print(f"{rxn_id} permanently removed: Not needed for growth.")

In [15]:
def merge_gpr_rxns(model, list_of_rxnpairs):
    for rxn_1, rxn_2 in list_of_rxnpairs:
        if (rxn_1 in model.reactions) and (rxn_2 in model.reactions):
            r1_obj = model.reactions.get_by_id(rxn_1)
            r2_obj = model.reactions.get_by_id(rxn_2)
            
            sol1 = model.slim_optimize()
            
            gpr1 = r1_obj.gene_reaction_rule.strip()
            gpr2 = r2_obj.gene_reaction_rule.strip()
            
            if gpr1 == gpr2:
                new_gpr = gpr1
            elif gpr1 and gpr2:
                new_gpr = f"({gpr1}) or ({gpr2})"
            else:
                new_gpr = gpr1 if gpr1 else gpr2
                
            r1_obj.gene_reaction_rule = new_gpr
            
            model.remove_reactions([r2_obj])
            
            print(f"Successfully merged {rxn_2} into {rxn_1}. New GPR: {new_gpr}")

## Diphosphate test summary
This gives you an overview for all reactions flagged by the diphosphate test throughout all your models

In [16]:
make_irreversible_ppi, flip_and_make_irreversible_ppi = create_diphsophate_overview(test_results_dir)

19 reactions should be Irreversible reactions:
FACOAL170_anteiso
GALT
FACOAL170_ISO
SERASr
FACOAL150_ISO
AADb
FACOALPHDCA
FACOAL140_ISO
ADK2_1
FORMCOAL
FACOAL150_anteiso
FACOAL160_ISO
UDPACGLP
NAPRT
FACOAL200
FACOAL180_2
ADK2
NNATr
APAT_1
-------------------------------------------------------------------------

1 reactions should be Flipped and irreversible reactions:
ORPT


In [17]:
#make_irreversible_ppi = ["AADb", "APAT_1", "NNATr", "UDPACGLP", "NAPRT", "OXACOAL", "ADK2_1"]
#flip_and_make_irreversible_ppi = []
stays_reversible = []

## Main  
This is not done. Feel free to investigate all other duplicate lists or whatever else

In [18]:
def duplicates_dictsummary(dict2store, model_id, list_duplicates):
    for pair in list_duplicates:
        if pair not in dict2store:
            dict2store[pair] = [model_id]
        elif pair in dict2store:
            dict2store[pair].append(model_id)

In [19]:
# delete duplicate reactions that are identical but do not have a GPR 
rxns2delete = ["BTS3r", "PSUDS", "UAGPT2_1", "PNCDC_1", "ORNTAC_1", "PUNP6", "TMDPK_1", "ADK2_1", "ARGORNt7", "PSCVT_1", "PACL", "DGUPP"]

In [20]:
# reactions that are identical but have different GPRs
rxns2mergegpr = [("GLYCS_I", "LGTHL"), ("HMGL", "HMGL_2"), ("CBPS", "CBPS_1"), ("4CMLCL_kt", "CMLDC"), ("THZPSN", "THZPSN_1"), ("SHSL2", "SHSL2r"), ("CYSS", "CYSS_2"), ("NTPP10", "NTPP10_1"),
                 ("NTPP2", "NTPP2_1"), ("NADDP", "NADDPp_1"), ("LIPATPT", "LIPATPT_2"), ("IZPN", "IZPN_1"), ("BUPN", "UPPN"), ("HISTP", "HISTP_1"), ("HEX", "HEX_1"),("ACSERHS", "AHSERL4"),
                ("CYSDS", "TRPAS1"), ("MCCC", "MCTC_1"), ("AMANK", "AMANK_1"), ("ACODA", "ACODA_1"), ("HEX1", "HEX1_1"), ("OGDE1","OXOADLR" ), ("GTPDPK", "GTPDPK_1"), ("NFORGLUAH", "NFORGLUAH2"),
                ("DHPD", "DHPM1"), ("PGLYCP", "PGLYCP_1"), ("ACPS1", "ACPS1_1"), ("NTPP1", "NTPP1_1"), ("NTD1", "NTD1_1"), ("DGNSK", "DGNSK_1"), ("PNTK", "PNTK_1"), ("NTPP8", "NTPP8_1"),
                ("HPROa","PY5CCR") ]

In [21]:
rxns2keep = [("LEUTA", "LEUTAi"), ("ACACT13", "ACACT4r"), ("HACD3", "HACD3i"), ("HACD5", "HACD5i"), ("INDOLEt2pp", "INDOLEt2rpp"), ("P5CRx", "PRO1x"),("SEAHCYSHYD", "SEAHCYSHYD_1"), ("SULRS", "SULR_1")]

In [22]:
# Iterate over all xml files in the model_dir and applies all functions
redox_duplicates_all = {}
exact_duplicates_all = {}
directional_duplicates_all = {}
coefficient_duplicates_all = {}

# Pre-fetch directory listings for better performance
files_in_model_dir = os.listdir(model_dir)
files_in_save_dir = os.listdir(model_dir_save)

for file in files_in_model_dir:
    if file.endswith('_mb1_mdr.xml'):

        # Remove the .xml suffix to generate only the model id
        model_id = file[:-4]

        # Prints the model that is investigated right now
        print(f"Model: {model_id}")

        if f"{model_id}_rdr_dp.xml" not in files_in_save_dir:

            # Parse the model
            model = read_sbml_model(os.path.join(model_dir, file))

            # Create duplicate lists for all four types of duplicates MACAW tests for
            exact_duplicates, directional_duplicates, coefficient_duplicates, redox_duplicates = get_all_duplicate_pairs(model_id)

            # Create dict of redox_duplicates
            duplicates_dictsummary(redox_duplicates_all, model_id, redox_duplicates)
            duplicates_dictsummary(exact_duplicates_all, model_id, exact_duplicates)
            duplicates_dictsummary(directional_duplicates_all, model_id, directional_duplicates)
            duplicates_dictsummary(coefficient_duplicates_all, model_id, coefficient_duplicates)

            # Creates a dict containing the GPR information for all reactions in the model
            gpr_list_dict = create_gpr_list_dict(model)

            # Apply fixes for Diphosphate reactions
            fix_diphophate_reactions()
            # Apply fixes for exact duplicates
            fix_exact_duplicates()
            # Apply fixes for directional duplicates
            fix_directional_duplicates()
    
            # MANUALLY FIX not fixable reactions, check growth to see if deletion of reaction stops growth 
            if "5DH4DGLCD" in model.reactions:
                model.remove_reactions([model.reactions.get_by_id("5DH4DGLCD")])
            sol = model.slim_optimize()
            #print(f"{sol} 5DH")


            if "FPRA" in model.reactions:
                sol1 = model.slim_optimize()
                
                with model:
                    model.remove_reactions([model.reactions.get_by_id("FPRA")])
                    sol2 = model.slim_optimize()
                    print(f"FPRA Test - Before: {sol1:.4f}, After: {sol2:.4f}")
                
                # If sol2 is less than 10% of sol1, it means FPRA is CRITICAL for growth -> Keep it.
                if sol2 < (sol1 * 0.1):
                    print("FPRA kept: It is required for biomass growth.")
                else:
                    # Otherwise, growth is fine without it -> Delete it permanently.
                    model.remove_reactions([model.reactions.get_by_id("FPRA")])
                    print("FPRA permanently removed: Not needed for growth.")

            if "GLBRAN3" in model.reactions:
                model.remove_reactions([model.reactions.get_by_id("GLBRAN3")])
            sol = model.slim_optimize()
            #print(f"{sol} GL")

            if "ORNTAC_1" in model.reactions:
                model.remove_reactions([model.reactions.get_by_id("ORNTAC_1")])

            if "GLDBRAN3" in model.reactions:
                model.reactions.get_by_id("GLDBRAN3").bounds = (-1000, 1000)

            manual_delete_rxns(model, rxns2delete)
            merge_gpr_rxns(model, rxns2mergegpr)
            print("-----------------")
            # Save model
            write_sbml_model(model, os.path.join(model_dir_save, f"{model_id}_rdr_dp.xml"))

Model: 644_or_mb1_mdr
Fixed ORPT in m_644_
No fix possible for GTPDPK and GTPDPK_1 with status: Different lenght of GPR list and share some genes
No fix possible for ACACT6r and ACACT9 with status: Different lenght of GPR list but no common genes
PSUDS Test - Before: 54.4103, After: 54.4103
PSUDS permanently removed: Not needed for growth.
Successfully merged HMGL_2 into HMGL. New GPR: (JMCDPC_00906 or JMCDPC_04631) or (JMCDPC_00417 or JMCDPC_00981 or JMCDPC_01150 or JMCDPC_01157 or JMCDPC_04640)
Successfully merged CMLDC into 4CMLCL_kt. New GPR: (JMCDPC_02701) or (JMCDPC_00697)
Successfully merged CYSS_2 into CYSS. New GPR: (JMCDPC_01204 or JMCDPC_01844 or JMCDPC_02144 or JMCDPC_02338) or (JMCDPC_01204)
Successfully merged UPPN into BUPN. New GPR: (JMCDPC_00220) or (JMCDPC_01313)
Successfully merged AHSERL4 into ACSERHS. New GPR: (JMCDPC_01204) or (JMCDPC_01204 or JMCDPC_01844)
Successfully merged AMANK_1 into AMANK. New GPR: (JMCDPC_01645) or (JMCDPC_00023)
Successfully merged OXOADL

### Manual fixes

In [322]:
loaded_model_save = {}
for file in os.listdir(model_dir_save):
    if not file.endswith(('.xml', '.sbml')):
        continue
        
    model = read_sbml_model(os.path.join(model_dir_save, file))
    model_id = model.id
    loaded_model_save[model_id] = model

In [324]:
# Create duplicate lists for all four types of duplicates MACAW tests for
for model_id, model in loaded_model_save.items():
        model_id = model_id[2:] + "or_mb1_mdr"
        exact_duplicates, directional_duplicates, coefficient_duplicates, redox_duplicates = get_all_duplicate_pairs(model_id)

        # Create dict of redox_duplicates
        duplicates_dictsummary(redox_duplicates_all, model_id, redox_duplicates)
        duplicates_dictsummary(exact_duplicates_all, model_id, exact_duplicates)
        duplicates_dictsummary(directional_duplicates_all, model_id, directional_duplicates)
        duplicates_dictsummary(coefficient_duplicates_all, model_id, coefficient_duplicates)

        # Creates a dict containing the GPR information for all reactions in the model
        gpr_list_dict = create_gpr_list_dict(model)

        # Apply fixes for Diphosphate reactions
        fix_diphophate_reactions()
        # Apply fixes for exact duplicates
        fix_exact_duplicates()
        # Apply fixes for directional duplicates
        fix_directional_duplicates()

        # MANUALLY FIX not fixable reactions, check growth to see if deletion of reaction stops growth 
        if "5DH4DGLCD" in model.reactions:
            model.remove_reactions([model.reactions.get_by_id("5DH4DGLCD")])
        sol = model.slim_optimize()
        #print(f"{sol} 5DH")


        if "FPRA" in model.reactions:
            sol1 = model.slim_optimize()
            
            with model:
                model.remove_reactions([model.reactions.get_by_id("FPRA")])
                sol2 = model.slim_optimize()
                print(f"FPRA Test - Before: {sol1:.4f}, After: {sol2:.4f}")
            
            # If sol2 is less than 10% of sol1, it means FPRA is CRITICAL for growth -> Keep it.
            if sol2 < (sol1 * 0.1):
                print("FPRA kept: It is required for biomass growth.")
            else:
                # Otherwise, growth is fine without it -> Delete it permanently.
                model.remove_reactions([model.reactions.get_by_id("FPRA")])
                print("FPRA permanently removed: Not needed for growth.")

        if "GLBRAN3" in model.reactions:
            model.remove_reactions([model.reactions.get_by_id("GLBRAN3")])
        sol = model.slim_optimize()
        #print(f"{sol} GL")

        if "GLDBRAN3" in model.reactions:
            model.reactions.get_by_id("GLDBRAN3").bounds = (-1000, 1000)

        manual_delete_rxns(model, rxns2delete)
        merge_gpr_rxns(model, rxns2mergegpr)
        print("-----------------")

Fixed ORPT in m_262_
No fix possible for SEAHCYSHYD and SEAHCYSHYD_1 with status: No GPR but one not a boundary reaction
-----------------
Fixed ORPT in m_397_
No fix possible for NADDP and NADDPp_1 with status: One or both reactions missing from GPR dict
-----------------
Fixed ORPT in m_1334_
No fix possible for HACD5 and HACD5i with status: Same length of GPR list and share some genes
-----------------
Fixed ORPT in m_428_
No fix possible for HACD5 and HACD5i with status: Different lenght of GPR list but no common genes
-----------------
Fixed ORPT in m_946_
No fix possible for SULR and SULR_1 with status: Different lenght of GPR list but no common genes
-----------------
Fixed ORPT in m_1208_
No fix possible for HACD5 and HACD5i with status: Different lenght of GPR list and share some genes
-----------------
Fixed ORPT in m_2872_
No fix possible for HACD5 and HACD5i with status: Different lenght of GPR list and share some genes
-----------------
Fixed ORPT in m_161_
No fix possible